# Document Analysis & Visualization

Load a PDF or CSV, extract structured data via LLM, render interactive Plotly charts.

**Providers:** OpenAI · Anthropic · Ollama — switch with a single variable.

In [ ]:
%pip install pymupdf plotly python-dotenv \
             langchain-openai langchain-anthropic langchain-ollama \
             langchain-community

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()  # loads from .env file if present

# ── Choose your provider ──────────────────────────────
LLM_PROVIDER = "openai"   # "openai" | "anthropic" | "ollama"

# API keys — set here or in a .env file
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY", "sk-...")
ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY", "sk-ant-...")
# Ollama needs no API key — just run: `ollama pull llama3.2`

In [ ]:
def build_llm(provider: str):
    if provider == "openai":
        from langchain_openai import ChatOpenAI
        os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
        return ChatOpenAI(model="gpt-4o-mini", temperature=0)

    elif provider == "anthropic":
        from langchain_anthropic import ChatAnthropic
        os.environ["ANTHROPIC_API_KEY"] = ANTHROPIC_API_KEY
        return ChatAnthropic(model="claude-sonnet-4-6", temperature=0)

    elif provider == "ollama":
        from langchain_ollama import ChatOllama
        return ChatOllama(model="llama3.2", temperature=0)

    else:
        raise ValueError(f"Unknown provider: {provider}. Use 'openai', 'anthropic', or 'ollama'.")

llm = build_llm(LLM_PROVIDER)
print(f"LLM ready: {LLM_PROVIDER}")

In [ ]:
from pathlib import Path
from typing import List, Optional

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from pydantic import BaseModel, Field

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.document_loaders.csv_loader import CSVLoader
from langchain_core.prompts import ChatPromptTemplate

In [ ]:
class DataPoint(BaseModel):
    label: str = Field(description="Name or label of the data point")
    value: float = Field(description="Numeric value")
    category: str = Field(description="Category or group this belongs to")
    unit: Optional[str] = Field(default=None, description="Unit of measurement if present")

class ExtractedReport(BaseModel):
    title: str = Field(description="Title or topic of the document")
    summary: str = Field(description="2-3 sentence summary of key findings")
    data_points: List[DataPoint] = Field(description="All numerical data found")
    key_metrics: dict = Field(description="Top-level KPIs as name:value pairs")

In [ ]:
def load_document(file_path: str) -> str:
    path = Path(file_path)
    if path.suffix.lower() == ".pdf":
        docs = PyMuPDFLoader(file_path).load()
        return "\n\n".join(d.page_content for d in docs)
    elif path.suffix.lower() == ".csv":
        docs = CSVLoader(file_path).load()
        return "\n".join(d.page_content for d in docs)
    else:
        raise ValueError(f"Unsupported format: {path.suffix}. Use .pdf or .csv")

print("Document loader ready")

In [ ]:
structured_llm = llm.with_structured_output(ExtractedReport)

prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a data extraction specialist.
Analyze the document and extract ALL numerical data points, metrics, and statistics.
Be precise with values. Capture units when present."""),
    ("human", "Document content:\n\n{content}\n\nExtract all structured data.")
])

extraction_chain = prompt | structured_llm
print("Extraction chain ready")

In [ ]:
FILE_PATH = "your_document.pdf"   # <- change to your file

content = load_document(FILE_PATH)
print(f"Loaded {len(content):,} characters from {FILE_PATH}")

result: ExtractedReport = extraction_chain.invoke({"content": content})

print(f"\nTitle:        {result.title}")
print(f"Summary:      {result.summary}")
print(f"Data points:  {len(result.data_points)}")
print(f"Key metrics:  {list(result.key_metrics.keys())}")

In [ ]:
df = pd.DataFrame([
    {"label": dp.label, "value": dp.value, "category": dp.category, "unit": dp.unit}
    for dp in result.data_points
])
print(df.to_string(index=False))

In [ ]:
fig = px.bar(
    df, x="label", y="value", color="category",
    title=f"{result.title} — Values by Label",
    text_auto=True, height=450
)
fig.update_layout(xaxis_tickangle=-40)
fig.show()

In [ ]:
cat_totals = df.groupby("category")["value"].sum().reset_index()

fig = px.pie(
    cat_totals, values="value", names="category",
    title=f"{result.title} — Distribution by Category",
    hole=0.35
)
fig.show()

In [ ]:
metrics_df = pd.DataFrame(
    list(result.key_metrics.items()), columns=["Metric", "Value"]
)

fig = go.Figure(data=[go.Table(
    header=dict(
        values=["Metric", "Value"],
        fill_color="#4C78A8",
        font=dict(color="white", size=13)
    ),
    cells=dict(values=[metrics_df["Metric"], metrics_df["Value"]])
)])
fig.update_layout(title="Key Metrics Summary")
fig.show()

In [ ]:
print("=" * 60)
print(f"  {result.title}")
print("=" * 60)
print(f"\n{result.summary}")
print(f"\nCategories : {df['category'].unique().tolist()}")
print(f"Data points: {len(df)}")